In [ ]:
import importlib.util
import sys

print("Python version:", sys.version.split()[0])

for package in ["pandas", "openpyxl", "plotly"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if available else 'Not installed'}")

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

print("Libraries imported successfully")
print("Current folder:", Path.cwd())

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
data_path = PROJECT_ROOT / "data" / "AdventureWorks Sales.xlsx"

print("File exists:", data_path.is_file())

if data_path.is_file():
    with pd.ExcelFile(data_path, engine="openpyxl") as workbook:
        print("\nSheet names:")
        for name in workbook.sheet_names:
            print("-", name)

In [ ]:
tables = pd.read_excel(
    data_path,
    sheet_name=None,
    engine="openpyxl"
)

table_summary = pd.DataFrame([
    {
        "Sheet": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in tables.items()
])

display(table_summary)

In [ ]:
sales = tables["Sales_data"].copy()

with pd.option_context("display.max_columns", None):
    display(sales.head(5))

In [ ]:
sales_check = pd.DataFrame({
    "Data Type": sales.dtypes.astype(str),
    "Missing Values": sales.isna().sum()
})

display(sales_check)

print("Fully duplicated rows:", sales.duplicated().sum())
print(
    "Duplicated sales line keys:",
    sales["SalesOrderLineKey"].duplicated().sum()
)

In [ ]:
sales["OrderDate"] = pd.to_datetime(
    sales["OrderDateKey"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

display(sales[["OrderDateKey", "OrderDate"]].head())

print("Invalid or missing order dates:", sales["OrderDate"].isna().sum())
print("First order date:", sales["OrderDate"].min())
print("Last order date:", sales["OrderDate"].max())

In [ ]:
numeric_columns = [
    "Order Quantity",
    "Unit Price",
    "Sales Amount",
    "Total Product Cost"
]

numeric_check = pd.DataFrame({
    "Minimum": sales[numeric_columns].min(),
    "Maximum": sales[numeric_columns].max(),
    "Zero Values": sales[numeric_columns].eq(0).sum(),
    "Negative Values": sales[numeric_columns].lt(0).sum()
})

display(numeric_check)

In [ ]:
sales["GrossProfit"] = (
    sales["Sales Amount"] - sales["Total Product Cost"]
)

display(
    sales[
        ["Sales Amount", "Total Product Cost", "GrossProfit"]
    ].head()
)

total_sales = sales["Sales Amount"].sum()
total_cost = sales["Total Product Cost"].sum()
total_profit = sales["GrossProfit"].sum()
gross_margin = total_profit / total_sales

print(f"Total sales: {total_sales:,.2f}")
print(f"Total product cost: {total_cost:,.2f}")
print(f"Gross profit: {total_profit:,.2f}")
print(f"Gross margin: {gross_margin:.2%}")
print("Loss-making sales lines:", sales["GrossProfit"].lt(0).sum())

In [ ]:
products = tables["Product_data"].copy()

with pd.option_context("display.max_columns", None):
    display(products.head())

print("Missing product keys:", products["ProductKey"].isna().sum())
print("Duplicate product keys:", products["ProductKey"].duplicated().sum())

print(
    "Sales lines without a matching product:",
    (~sales["ProductKey"].isin(products["ProductKey"])).sum()
)

In [ ]:
sales_enriched = sales.merge(
    products[["ProductKey", "Product", "Subcategory", "Category"]],
    on="ProductKey",
    how="left",
    validate="many_to_one",
    indicator="ProductMatch"
)

print("Rows before join:", len(sales))
print("Rows after join:", len(sales_enriched))
print(
    "Unmatched sales lines:",
    sales_enriched["ProductMatch"].eq("left_only").sum()
)

print(
    "Sales total unchanged:",
    abs(
        sales_enriched["Sales Amount"].sum()
        - sales["Sales Amount"].sum()
    ) < 0.01
)

display(
    sales_enriched[
        ["ProductKey", "Product", "Category", "Sales Amount", "GrossProfit"]
    ].head()
)

In [ ]:
territories = tables["Sales Territory_data"].copy()

display(territories)

print(
    "Missing territory keys:",
    territories["SalesTerritoryKey"].isna().sum()
)

print(
    "Duplicate territory keys:",
    territories["SalesTerritoryKey"].duplicated().sum()
)

print(
    "Sales lines without a matching territory:",
    (~sales_enriched["SalesTerritoryKey"].isin(
        territories["SalesTerritoryKey"]
    )).sum()
)

In [ ]:
sales_geo = sales_enriched.merge(
    territories,
    on="SalesTerritoryKey",
    how="left",
    validate="many_to_one",
    indicator="TerritoryMatch"
)

print("Rows before join:", len(sales_enriched))
print("Rows after join:", len(sales_geo))

print(
    "Unmatched sales lines:",
    sales_geo["TerritoryMatch"].eq("left_only").sum()
)

for column in ["Sales Amount", "Total Product Cost", "GrossProfit"]:
    unchanged = abs(
        sales_geo[column].sum() - sales_enriched[column].sum()
    ) < 0.01
    print(f"{column} total unchanged:", unchanged)

display(
    sales_geo[
        ["Product", "Region", "Country", "Sales Amount", "GrossProfit"]
    ].head()
)

In [ ]:
sales_orders = tables["Sales Order_data"].copy()

display(sales_orders.head())

print("\nMissing values:")
display(
    sales_orders[
        ["SalesOrderLineKey", "Sales Order", "Channel"]
    ].isna().sum()
)

print(
    "Duplicate sales line keys:",
    sales_orders["SalesOrderLineKey"].duplicated().sum()
)

print(
    "Sales lines without matching order information:",
    (~sales_geo["SalesOrderLineKey"].isin(
        sales_orders["SalesOrderLineKey"]
    )).sum()
)

print("\nSales lines by channel:")
display(sales_orders["Channel"].value_counts(dropna=False))

In [ ]:
sales_final = sales_geo.merge(
    sales_orders[["SalesOrderLineKey", "Sales Order", "Channel"]],
    on="SalesOrderLineKey",
    how="left",
    validate="one_to_one",
    indicator="OrderMatch"
)

print("Rows before join:", len(sales_geo))
print("Rows after join:", len(sales_final))
print(
    "Unmatched sales lines:",
    sales_final["OrderMatch"].eq("left_only").sum()
)

for column in ["Sales Amount", "Total Product Cost", "GrossProfit"]:
    unchanged = abs(
        sales_final[column].sum() - sales_geo[column].sum()
    ) < 0.01
    print(f"{column} total unchanged:", unchanged)

print("Distinct orders:", sales_final["Sales Order"].nunique())

display(
    sales_final[
        ["Sales Order", "Channel", "Product", "Country", "Sales Amount"]
    ].head()
)

In [ ]:
revenue = sales_final["Sales Amount"].sum()
product_cost = sales_final["Total Product Cost"].sum()
gross_profit = sales_final["GrossProfit"].sum()

order_count = sales_final["Sales Order"].nunique()
units_sold = sales_final["Order Quantity"].sum()

average_order_value = revenue / order_count
gross_margin = gross_profit / revenue

print(f"Total sales: {revenue:,.2f}")
print(f"Total product cost: {product_cost:,.2f}")
print(f"Gross profit: {gross_profit:,.2f}")
print(f"Gross margin: {gross_margin:.2%}")
print(f"Total orders: {order_count:,}")
print(f"Units sold: {units_sold:,}")
print(f"Average order value: {average_order_value:,.2f}")

In [ ]:
channel_summary = (
    sales_final.groupby("Channel", dropna=False)
    .agg(
        Revenue=("Sales Amount", "sum"),
        ProductCost=("Total Product Cost", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        Orders=("Sales Order", "nunique"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

channel_summary["GrossMarginPct"] = (
    channel_summary["GrossProfit"] / channel_summary["Revenue"] * 100
)

channel_summary["AverageOrderValue"] = (
    channel_summary["Revenue"] / channel_summary["Orders"]
)

channel_summary["RevenueSharePct"] = (
    channel_summary["Revenue"] / revenue * 100
)

display(channel_summary.round(2))

In [ ]:
channel_chart_data = channel_summary.melt(
    id_vars="Channel",
    value_vars=["Revenue", "GrossProfit"],
    var_name="Metric",
    value_name="Amount"
)

fig = px.bar(
    channel_chart_data,
    x="Channel",
    y="Amount",
    color="Metric",
    barmode="group",
    title="Sales and Gross Profit by Channel",
    labels={
        "Channel": "Sales channel",
        "Amount": "Amount",
        "Metric": "Measure"
    },
    color_discrete_map={
        "Revenue": "#2563EB",
        "GrossProfit": "#16A34A"
    },
    template="plotly_white"
)

fig.update_layout(
    yaxis_tickformat="~s",
    hovermode="closest"
)

fig.update_traces(
    hovertemplate="%{x}<br>Amount: %{y:,.2f}<extra>%{fullData.name}</extra>"
)

fig.show()

In [ ]:
fig.show(renderer="notebook")

In [ ]:
category_channel_summary = (
    sales_final.groupby(["Channel", "Category"], dropna=False)
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

category_channel_summary["GrossMarginPct"] = (
    category_channel_summary["GrossProfit"]
    / category_channel_summary["Revenue"]
    * 100
)

display(category_channel_summary.round(2))

In [ ]:
fig_margin = px.bar(
    category_channel_summary,
    x="Category",
    y="GrossMarginPct",
    color="Channel",
    barmode="group",
    text="GrossMarginPct",
    title="Gross Margin by Category and Sales Channel",
    labels={"GrossMarginPct": "Gross margin (%)"},
    color_discrete_map={
        "Internet": "#2563EB",
        "Reseller": "#F97316"
    },
    template="plotly_white"
)

fig_margin.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "%{x}<br>Gross margin: %{y:.2f}%"
        "<extra>%{fullData.name}</extra>"
    )
)

fig_margin.add_hline(y=0, line_color="black", line_width=1)
fig_margin.update_layout(yaxis_ticksuffix="%", margin=dict(t=80))

fig_margin.show(renderer="notebook")

In [ ]:
reseller_bikes = sales_final.loc[
    (sales_final["Channel"] == "Reseller")
    & (sales_final["Category"] == "Bikes")
].copy()

bike_subcategory_summary = (
    reseller_bikes
    .groupby("Subcategory", dropna=False)
    .agg(
        Revenue=("Sales Amount", "sum"),
        ProductCost=("Total Product Cost", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

bike_subcategory_summary["GrossMarginPct"] = (
    bike_subcategory_summary["GrossProfit"]
    / bike_subcategory_summary["Revenue"]
    * 100
)

bike_subcategory_summary = bike_subcategory_summary.sort_values(
    "GrossProfit"
)

display(bike_subcategory_summary.round(2))

In [ ]:
fig_bikes = px.bar(
    bike_subcategory_summary,
    x="GrossProfit",
    y="Subcategory",
    orientation="h",
    color="GrossProfit",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    text="GrossProfit",
    custom_data=["Revenue", "GrossMarginPct"],
    title="Reseller Bikes: Gross Profit by Subcategory",
    labels={
        "GrossProfit": "Gross profit (amount)",
        "Subcategory": "Bike subcategory"
    },
    template="plotly_white"
)

fig_bikes.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "%{y}<br>"
        "Revenue: %{customdata[0]:,.2f}<br>"
        "Gross profit: %{x:,.2f}<br>"
        "Gross margin: %{customdata[1]:.2f}%"
        "<extra></extra>"
    )
)

fig_bikes.add_vline(x=0, line_color="black", line_width=1)

fig_bikes.update_layout(
    coloraxis_showscale=False,
    xaxis_tickformat="~s",
    margin=dict(l=140, r=140),
    yaxis=dict(
        categoryorder="array",
        categoryarray=bike_subcategory_summary["Subcategory"].tolist(),
        autorange="reversed"
    )
)

fig_bikes.show(renderer="notebook")

In [ ]:
fig_bikes.update_traces(
    textposition="inside",
    insidetextanchor="middle",
    textfont=dict(color="white", size=14)
)

fig_bikes.update_xaxes(
    range=[
        bike_subcategory_summary["GrossProfit"].min() * 1.20,
        bike_subcategory_summary["GrossProfit"].max() * 1.20
    ]
)

fig_bikes.show(renderer="notebook")

In [ ]:
monthly_summary = (
    sales_final
    .groupby(pd.Grouper(key="OrderDate", freq="MS"))
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        Orders=("Sales Order", "nunique"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

monthly_summary["GrossMarginPct"] = (
    monthly_summary["GrossProfit"]
    / monthly_summary["Revenue"]
    * 100
)

print(
    "Data coverage:",
    sales_final["OrderDate"].min().date(),
    "to",
    sales_final["OrderDate"].max().date()
)

display(monthly_summary.round(2))

In [ ]:
last_date = sales_final["OrderDate"].max()
last_month_start = last_date.to_period("M").to_timestamp()

monthly_chart_data = monthly_summary.copy()

if not last_date.is_month_end:
    monthly_chart_data = monthly_chart_data.loc[
        monthly_chart_data["OrderDate"] < last_month_start
    ].copy()

fig_monthly = px.line(
    monthly_chart_data,
    x="OrderDate",
    y=["Revenue", "GrossProfit"],
    markers=True,
    title="Monthly Revenue and Gross Profit — through May 2020",
    labels={
        "OrderDate": "Month",
        "value": "Amount",
        "variable": "Measure"
    },
    color_discrete_map={
        "Revenue": "#2563EB",
        "GrossProfit": "#16A34A"
    },
    template="plotly_white"
)

fig_monthly.add_hline(
    y=0, line_color="gray", line_width=1
)

fig_monthly.update_layout(
    hovermode="x unified",
    yaxis_tickformat="~s",
    xaxis=dict(
        tickformat="%b %Y",
        rangeslider=dict(visible=True)
    )
)

fig_monthly.show(renderer="notebook")

In [ ]:
june_2018 = sales_final.loc[
    (sales_final["OrderDate"] >= "2018-06-01")
    & (sales_final["OrderDate"] < "2018-07-01")
].copy()

june_2018_summary = (
    june_2018
    .groupby(["Channel", "Category"], dropna=False)
    .agg(
        Revenue=("Sales Amount", "sum"),
        ProductCost=("Total Product Cost", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

june_2018_summary["GrossMarginPct"] = (
    june_2018_summary["GrossProfit"]
    / june_2018_summary["Revenue"]
    * 100
)

display(
    june_2018_summary
    .sort_values("GrossProfit")
    .round(2)
)

In [ ]:
june_reseller_bikes = june_2018.loc[
    (june_2018["Channel"] == "Reseller")
    & (june_2018["Category"] == "Bikes")
].copy()

june_reseller_bikes["RevenuePerUnit"] = (
    june_reseller_bikes["Sales Amount"]
    / june_reseller_bikes["Order Quantity"]
)

june_reseller_bikes["CostPerUnit"] = (
    june_reseller_bikes["Total Product Cost"]
    / june_reseller_bikes["Order Quantity"]
)

columns_to_check = [
    "Sales Order",
    "ProductKey",
    "Product",
    "Subcategory",
    "Order Quantity",
    "Unit Price",
    "Unit Price Discount Pct",
    "Sales Amount",
    "Total Product Cost",
    "RevenuePerUnit",
    "CostPerUnit",
    "GrossProfit"
]

display(
    june_reseller_bikes
    .sort_values("GrossProfit")
    [columns_to_check]
    .head(10)
    .round(2)
)

In [ ]:
pricing_check = june_reseller_bikes.copy()

pricing_check["PriceTimesQty"] = (
    pricing_check["Unit Price"]
    * pricing_check["Order Quantity"]
)

pricing_check["ExpectedSales"] = (
    pricing_check["PriceTimesQty"]
    * (1 - pricing_check["Unit Price Discount Pct"])
)

pricing_check["SalesGap"] = (
    pricing_check["Sales Amount"]
    - pricing_check["ExpectedSales"]
)

pricing_check["ImpliedReductionPct"] = (
    1
    - pricing_check["Sales Amount"]
    / pricing_check["PriceTimesQty"]
) * 100

display(
    pricing_check.sort_values("GrossProfit")[
        [
            "Product",
            "Order Quantity",
            "Unit Price",
            "Unit Price Discount Pct",
            "Extended Amount",
            "PriceTimesQty",
            "ExpectedSales",
            "Sales Amount",
            "SalesGap",
            "ImpliedReductionPct"
        ]
    ].head(10).style.format(precision=4)
)

In [ ]:
pricing_check["ReductionPctRounded"] = (
    pricing_check["ImpliedReductionPct"].round(2)
)

reduction_summary = (
    pricing_check
    .groupby("ReductionPctRounded", dropna=False)
    .agg(
        SalesLines=("SalesOrderLineKey", "count"),
        Revenue=("Sales Amount", "sum"),
        ProductCost=("Total Product Cost", "sum"),
        GrossProfit=("GrossProfit", "sum")
    )
    .reset_index()
)

print("Recorded discount values:")
print(
    pricing_check["Unit Price Discount Pct"]
    .value_counts(dropna=False)
)

display(reduction_summary.round(2))

In [ ]:
pricing_audit = sales_final.copy()

pricing_audit["ImpliedReductionPct"] = (
    1
    - pricing_audit["Sales Amount"]
    / (
        pricing_audit["Unit Price"]
        * pricing_audit["Order Quantity"]
    )
) * 100

pricing_audit["ReductionPctRounded"] = (
    pricing_audit["ImpliedReductionPct"].round(2)
)

pricing_audit_summary = (
    pricing_audit
    .groupby(
        ["Channel", "ReductionPctRounded"],
        dropna=False
    )
    .agg(
        SalesLines=("SalesOrderLineKey", "count"),
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        RecordedDiscountMin=("Unit Price Discount Pct", "min"),
        RecordedDiscountMax=("Unit Price Discount Pct", "max")
    )
    .reset_index()
)

display(pricing_audit_summary.round(2))

In [ ]:
country_summary = (
    sales_final
    .groupby("Country", dropna=False)
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        Orders=("Sales Order", "nunique"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

country_summary["GrossMarginPct"] = (
    country_summary["GrossProfit"]
    / country_summary["Revenue"]
    * 100
)

country_summary["RevenueSharePct"] = (
    country_summary["Revenue"]
    / sales_final["Sales Amount"].sum()
    * 100
)

display(
    country_summary
    .sort_values("Revenue", ascending=False)
    .round(2)
)

In [ ]:
fig_country = px.scatter(
    country_summary,
    x="Revenue",
    y="GrossMarginPct",
    size="GrossProfit",
    text="Country",
    hover_name="Country",
    hover_data={
        "Revenue": ":,.2f",
        "GrossProfit": ":,.2f",
        "GrossMarginPct": ":.2f",
        "RevenueSharePct": ":.2f"
    },
    size_max=65,
    title="Country Performance: Revenue vs Gross Margin",
    labels={
        "Revenue": "Revenue (amount)",
        "GrossMarginPct": "Gross margin (%)",
        "GrossProfit": "Gross profit",
        "RevenueSharePct": "Revenue share (%)"
    },
    template="plotly_white"
)

fig_country.update_traces(
    textposition="top center",
    marker=dict(color="#2563EB", opacity=0.75)
)

fig_country.update_layout(
    xaxis_tickformat="~s",
    yaxis_ticksuffix="%",
    margin=dict(t=90, r=130)
)

fig_country.show(renderer="notebook")

In [ ]:
country_channel_summary = (
    sales_final
    .groupby(["Country", "Channel"], dropna=False)
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        Orders=("Sales Order", "nunique")
    )
    .reset_index()
)

country_channel_summary["GrossMarginPct"] = (
    country_channel_summary["GrossProfit"]
    / country_channel_summary["Revenue"]
    * 100
)

country_channel_summary["ChannelRevenueSharePct"] = (
    country_channel_summary["Revenue"]
    / country_channel_summary.groupby("Country")[
        "Revenue"
    ].transform("sum")
    * 100
)

display(
    country_channel_summary
    .sort_values(["Country", "Channel"])
    .round(2)
)

In [ ]:
fig_channel_mix = px.bar(
    country_channel_summary,
    x="Country",
    y="ChannelRevenueSharePct",
    color="Channel",
    text="ChannelRevenueSharePct",
    barmode="stack",
    title="Revenue Mix by Country and Sales Channel",
    labels={
        "Country": "Country",
        "ChannelRevenueSharePct": "Share of country revenue (%)"
    },
    color_discrete_map={
        "Internet": "#2563EB",
        "Reseller": "#F97316"
    },
    template="plotly_white"
)

fig_channel_mix.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="inside",
    hovertemplate=(
        "%{x}<br>"
        "Revenue share: %{y:.2f}%"
        "<extra>%{fullData.name}</extra>"
    )
)

fig_channel_mix.update_layout(
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    legend_title_text="Sales channel"
)

fig_channel_mix.show(renderer="notebook")

In [ ]:
product_summary = (
    sales_final
    .groupby(
        ["Product", "Category", "Subcategory"],
        dropna=False
    )
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        UnitsSold=("Order Quantity", "sum")
    )
    .reset_index()
)

product_summary["GrossMarginPct"] = (
    product_summary["GrossProfit"]
    / product_summary["Revenue"]
    * 100
)

top_profit_products = (
    product_summary
    .sort_values("GrossProfit", ascending=False)
    .head(10)
)

display(top_profit_products.round(2))

In [ ]:
fig_top_products = px.bar(
    top_profit_products,
    x="GrossProfit",
    y="Product",
    orientation="h",
    text="GrossProfit",
    custom_data=["Revenue", "GrossMarginPct", "UnitsSold"],
    title="Top 10 Products by Gross Profit — All Channels",
    labels={
        "GrossProfit": "Gross profit (amount)",
        "Product": "Product"
    },
    template="plotly_white"
)

fig_top_products.update_traces(
    marker_color="#2563EB",
    texttemplate="%{text:,.0f}",
    textposition="inside",
    textfont_color="white",
    hovertemplate=(
        "%{y}<br>"
        "Gross profit: %{x:,.2f}<br>"
        "Revenue: %{customdata[0]:,.2f}<br>"
        "Gross margin: %{customdata[1]:.2f}%<br>"
        "Units sold: %{customdata[2]:,}"
        "<extra></extra>"
    )
)

fig_top_products.update_layout(
    height=650,
    xaxis_tickformat="~s",
    yaxis=dict(
        categoryorder="array",
        categoryarray=top_profit_products["Product"].tolist(),
        autorange="reversed"
    ),
    margin=dict(l=230, r=50)
)

fig_top_products.show(renderer="notebook")

In [ ]:
loss_products = (
    product_summary.loc[
        product_summary["GrossProfit"] < 0
    ]
    .sort_values("GrossProfit")
)

print("Number of products with overall gross loss:", len(loss_products))

print(
    "Combined gross loss of these products:",
    f'{loss_products["GrossProfit"].sum():,.2f}'
)

display(loss_products.head(10).round(2))

In [ ]:
top_loss_products = loss_products.head(10).copy()

fig_loss_products = px.bar(
    top_loss_products,
    x="GrossProfit",
    y="Product",
    orientation="h",
    text="GrossProfit",
    custom_data=["Revenue", "GrossMarginPct", "UnitsSold"],
    title="10 Products with the Largest Gross Loss — All Channels",
    labels={
        "GrossProfit": "Gross profit (negative = loss)",
        "Product": "Product"
    },
    template="plotly_white"
)

fig_loss_products.update_traces(
    marker_color="#DC2626",
    texttemplate="%{text:,.0f}",
    textposition="inside",
    textfont_color="white",
    hovertemplate=(
        "%{y}<br>"
        "Gross profit: %{x:,.2f}<br>"
        "Revenue: %{customdata[0]:,.2f}<br>"
        "Gross margin: %{customdata[1]:.2f}%<br>"
        "Units sold: %{customdata[2]:,}"
        "<extra></extra>"
    )
)

fig_loss_products.add_vline(
    x=0, line_color="black", line_width=1
)

fig_loss_products.update_layout(
    height=650,
    xaxis_tickformat="~s",
    yaxis=dict(
        categoryorder="array",
        categoryarray=top_loss_products["Product"].tolist(),
        autorange="reversed"
    ),
    margin=dict(l=260, r=50)
)

fig_loss_products.show(renderer="notebook")

In [ ]:
profitable_products_gp = product_summary.loc[
    product_summary["GrossProfit"] > 0, "GrossProfit"
].sum()

loss_products_gp = loss_products["GrossProfit"].sum()

reconciled_gp = profitable_products_gp + loss_products_gp
sales_gp = sales_final["GrossProfit"].sum()

profit_reconciliation = pd.DataFrame({
    "Measure": [
        "Gross profit from profitable product groups",
        "Gross loss from loss-making product groups",
        "Combined gross profit",
        "Gross profit from sales data",
        "Difference"
    ],
    "Amount": [
        profitable_products_gp,
        loss_products_gp,
        reconciled_gp,
        sales_gp,
        reconciled_gp - sales_gp
    ]
})

display(
    profit_reconciliation.style.format({
        "Amount": "{:,.2f}"
    })
)

In [ ]:
comparison_data = sales_final.loc[
    sales_final["OrderDate"].dt.year.isin([2019, 2020])
    & (sales_final["OrderDate"].dt.month <= 5)
].copy()

comparison_data["Year"] = (
    comparison_data["OrderDate"].dt.year
)

year_comparison = (
    comparison_data
    .groupby("Year")
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        Orders=("Sales Order", "nunique"),
        UnitsSold=("Order Quantity", "sum")
    )
    .sort_index()
)

year_comparison["GrossMarginPct"] = (
    year_comparison["GrossProfit"]
    / year_comparison["Revenue"]
    * 100
)

year_comparison["RevenueGrowthPct"] = (
    year_comparison["Revenue"].pct_change(fill_method=None) * 100
)

year_comparison["GrossProfitGrowthPct"] = (
    year_comparison["GrossProfit"].pct_change(fill_method=None) * 100
)

display(year_comparison.round(2))

In [ ]:
year_channel_comparison = (
    comparison_data
    .groupby(["Year", "Channel"])
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum"),
        Orders=("Sales Order", "nunique")
    )
    .reset_index()
)

year_channel_comparison["AverageOrderValue"] = (
    year_channel_comparison["Revenue"]
    / year_channel_comparison["Orders"]
)

year_channel_comparison["GrossMarginPct"] = (
    year_channel_comparison["GrossProfit"]
    / year_channel_comparison["Revenue"]
    * 100
)

year_channel_comparison["RevenueSharePct"] = (
    year_channel_comparison["Revenue"]
    / year_channel_comparison.groupby("Year")[
        "Revenue"
    ].transform("sum")
    * 100
)

display(year_channel_comparison.round(2))

In [ ]:
year_channel_chart = year_channel_comparison.melt(
    id_vars=["Year", "Channel"],
    value_vars=["Revenue", "GrossProfit"],
    var_name="Metric",
    value_name="Amount"
)

year_channel_chart["Year"] = (
    year_channel_chart["Year"].astype(str)
)

fig_year_channel = px.bar(
    year_channel_chart,
    x="Year",
    y="Amount",
    color="Channel",
    facet_col="Metric",
    barmode="group",
    title="January–May Performance: 2019 vs 2020",
    labels={"Amount": "Amount", "Year": "Year"},
    color_discrete_map={
        "Internet": "#2563EB",
        "Reseller": "#F97316"
    },
    template="plotly_white"
)

fig_year_channel.update_yaxes(
    matches=None,
    tickformat="~s",
    rangemode="tozero",
    showticklabels=True
)

fig_year_channel.update_traces(
    hovertemplate=(
        "Year: %{x}<br>"
        "Amount: %{y:,.2f}"
        "<extra>%{fullData.name}</extra>"
    )
)

fig_year_channel.show(renderer="notebook")

In [ ]:
reseller_year_category = (
    comparison_data.loc[
        comparison_data["Channel"] == "Reseller"
    ]
    .groupby(["Year", "Category"])
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum")
    )
    .reset_index()
)

reseller_year_category["GrossMarginPct"] = (
    reseller_year_category["GrossProfit"]
    / reseller_year_category["Revenue"]
    * 100
)

display(
    reseller_year_category
    .sort_values(["Category", "Year"])
    .round(2)
)

In [ ]:
reseller_year_category = (
    comparison_data.loc[
        comparison_data["Channel"] == "Reseller"
    ]
    .groupby(["Year", "Category"])
    .agg(
        Revenue=("Sales Amount", "sum"),
        GrossProfit=("GrossProfit", "sum")
    )
    .reset_index()
)

reseller_year_category["GrossMarginPct"] = (
    reseller_year_category["GrossProfit"]
    / reseller_year_category["Revenue"]
    * 100
)

display(
    reseller_year_category
    .sort_values(["Category", "Year"])
    .round(2)
)

In [ ]:
reseller_profit_change = (
    reseller_year_category
    .pivot(
        index="Category",
        columns="Year",
        values="GrossProfit"
    )
    .rename(columns={
        2019: "GrossProfit2019",
        2020: "GrossProfit2020"
    })
)

reseller_profit_change["ProfitChange"] = (
    reseller_profit_change["GrossProfit2020"]
    - reseller_profit_change["GrossProfit2019"]
)

reseller_profit_change = (
    reseller_profit_change
    .sort_values("ProfitChange")
    .reset_index()
)

display(reseller_profit_change.round(2))

print(
    "Total change in Reseller gross profit:",
    f'{reseller_profit_change["ProfitChange"].sum():,.2f}'
)

In [ ]:
import plotly.graph_objects as go

start_profit = reseller_year_category.loc[
    reseller_year_category["Year"] == 2019,
    "GrossProfit"
].sum()

end_profit = reseller_year_category.loc[
    reseller_year_category["Year"] == 2020,
    "GrossProfit"
].sum()

fig_profit_bridge = go.Figure(go.Waterfall(
    x=(
        ["2019 Jan–May"]
        + reseller_profit_change["Category"].tolist()
        + ["2020 Jan–May"]
    ),
    measure=["absolute"] + ["relative"] * 4 + ["total"],
    y=(
        [start_profit]
        + reseller_profit_change["ProfitChange"].tolist()
        + [0]
    ),
    text=[
        f"{value:,.0f}"
        for value in (
            [start_profit]
            + reseller_profit_change["ProfitChange"].tolist()
            + [end_profit]
        )
    ],
    textposition="outside",
    decreasing=dict(marker=dict(color="#DC2626")),
    increasing=dict(marker=dict(color="#16A34A")),
    totals=dict(marker=dict(color="#2563EB")),
    connector=dict(line=dict(color="#94A3B8"))
))

fig_profit_bridge.update_layout(
    title="Reseller Gross Profit Change: January–May 2019 to 2020",
    yaxis_title="Gross profit (amount)",
    yaxis_tickformat="~s",
    template="plotly_white",
    showlegend=False,
    margin=dict(t=90, b=80)
)

fig_profit_bridge.show(renderer="notebook")

### Reseller profitability: January–May 2019 vs 2020

Reseller revenue increased from 10.01 million to 12.62 million,
while gross profit fell from 391,472.31 to 1,530.77.
Gross margin declined from 3.91% to approximately 0.01%.

Bikes accounted for a gross profit decrease of 343,913.31,
equivalent to approximately 88.2% of the total net decline.
Components and Clothing also declined, while Accessories
partially offset these decreases with an improvement of 33,829.75.

These results identify where profitability deteriorated.
They do not establish whether pricing, product mix, costs,
or source-data issues caused the changes.

The recorded discount field is zero throughout the dataset,
despite differences between Sales Amount and Unit Price × Quantity.
Any inferred revenue reductions therefore require source validation.

## Key findings and business recommendations

- Reseller generated 73.26% of total revenue but achieved only
  a 0.58% gross margin. Internet achieved a 41.15% gross margin.

- During January–May, total revenue increased by 66.23%
  from 2019 to 2020, while gross profit increased by 131.85%.

- Despite overall growth, Reseller gross profit declined sharply.
  Bikes accounted for approximately 88.2% of its net decline.

- Prioritize a review of Reseller bike pricing, product costs,
  and product mix before recommending pricing changes.

- Investigate differences between recorded sales amounts
  and Unit Price × Quantity with the source-data owner.

## Limitations

- Gross profit includes product costs only; it is not net profit.
- June 2020 is incomplete in the available data.
  The year-over-year comparison therefore uses January–May.
- Recorded discounts are zero throughout the dataset.
  Inferred revenue reductions are not verified discounts.
- These findings describe patterns and do not establish causation.
- The dataset's currency has not been independently verified.

In [ ]:
dashboard_columns = [
    "SalesOrderLineKey",
    "Sales Order",
    "OrderDate",
    "Channel",
    "Country",
    "Region",
    "ProductKey",
    "Product",
    "Category",
    "Subcategory",
    "Order Quantity",
    "Unit Price",
    "Sales Amount",
    "Total Product Cost",
    "GrossProfit",
    "Unit Price Discount Pct",
]

dashboard_data = sales_final[dashboard_columns].copy()

print("Dashboard dataset shape:", dashboard_data.shape)
print(
    "Duplicate sales line keys:",
    dashboard_data["SalesOrderLineKey"].duplicated().sum()
)
print("\nMissing values:")
print(dashboard_data.isna().sum())

display(dashboard_data.head())

In [ ]:
from pathlib import Path

project_folder = PROJECT_ROOT
assert project_folder.is_dir(), "Project folder পাওয়া যায়নি—path চেক করো।"

processed_folder = project_folder / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

csv_path = processed_folder / "sales_dashboard.csv"

dashboard_data.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

print("CSV saved successfully!")
print("Location:", csv_path)
print("Rows:", len(dashboard_data))
print("File size (MB):", round(csv_path.stat().st_size / (1024 ** 2), 2))

In [ ]:
import numpy as np

saved_data = pd.read_csv(
    csv_path,
    encoding="utf-8-sig",
    parse_dates=["OrderDate"]
)

assert saved_data.shape == dashboard_data.shape
assert saved_data["SalesOrderLineKey"].is_unique
assert saved_data["OrderDate"].notna().all()

money_columns = ["Sales Amount", "Total Product Cost", "GrossProfit"]

np.testing.assert_allclose(
    saved_data[money_columns].sum(),
    dashboard_data[money_columns].sum(),
    rtol=0,
    atol=0.01
)

print("CSV validation passed!")
print("Shape:", saved_data.shape)
print(
    "Date range:",
    saved_data["OrderDate"].min().date(),
    "to",
    saved_data["OrderDate"].max().date()
)

print("\nVerified totals:")
print(saved_data[money_columns].sum().round(2))